# Notebook 08 — Cell Fate Mapping with CellRank

**Project:** RetinoblastomaAtlas  
**Input:** `data/processed/08_velocity.h5ad`  
**Output:** `data/processed/09_cellrank.h5ad`, macrostate UMAPs, fate probability plots, driver genes

---

## Scientific Background

RNA velocity (notebook 07) provides a **local** measure of transcriptional directionality. CellRank (Lange et al. 2022, *Nature Methods*) extends this to a **global probabilistic framework** for cell fate prediction:

1. Combines RNA velocity vectors with cell–cell similarity into a **Markov chain transition matrix** — robust fate predictions even for cells with noisy individual velocity vectors
2. Identifies **initial and terminal states** automatically using GPCCA (Generalized Perron Cluster Cluster Analysis)
3. Computes **fate probabilities**: for each cell, the probability of reaching each terminal state:
   - Terminal A: mature, non-invasive cone-like tumour (Subtype 1)
   - Terminal B: dedifferentiated, invasive tumour (Subtype 2 / extraocular)
4. Identifies **driver genes**: genes whose expression correlates with fate probability

## Significance

By comparing fate probabilities between intraocular and extraocular samples, we quantify whether extraocular tumour cells are preferentially committed to the invasive terminal state — providing mechanistic insight into why extraocular tumours are more aggressive.

## Kernel Combination Strategy

We use a **combined kernel**:
- `VelocityKernel` (weight 0.8): RNA velocity direction
- `ConnectivityKernel` (weight 0.2): scVI-based KNN similarity

The ConnectivityKernel provides stability for cells with low velocity confidence, preventing spurious fate assignments.

**References:**  
- Lange M et al. *Nat Methods* 2022. https://doi.org/10.1038/s41592-021-01346-6  
- Bergen V et al. *Nat Biotechnol* 2020. https://doi.org/10.1038/s41587-020-0591-3

## Parameters

In [ ]:
N_STATES = 4  # Number of macrostates for GPCCA
           #   Recommendation: start with 4 (cone-like, proliferating, invasive, immune),
           #   adjust if GPCCA finds too few/many meaningful states

## Setup

In [ ]:
import warnings
from pathlib import Path

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc

warnings.filterwarnings("ignore")
sc.settings.verbosity = 1

ROOT     = Path("..").resolve()
IN_H5AD  = ROOT / "data" / "processed" / "08_velocity.h5ad"
OUT_H5AD = ROOT / "data" / "processed" / "09_cellrank.h5ad"
FIG_DIR  = ROOT / "results" / "figures"
TAB_DIR  = ROOT / "results" / "tables"

FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

## Step 1 — Load Velocity-Annotated Atlas

In [ ]:
print("Step 1 — Loading velocity-annotated atlas")
adata = sc.read_h5ad(IN_H5AD)
print(f"  {adata.n_obs:,} cells x {adata.n_vars:,} genes")

# Import CellRank
import cellrank as cr
from cellrank.kernels import VelocityKernel, ConnectivityKernel
from cellrank.estimators import GPCCA

print(f"  CellRank version: {cr.__version__}")

## Step 2 — Build Combined Velocity + Connectivity Kernel

The combined kernel integrates:
- **VelocityKernel** (80%): uses RNA velocity cosine similarities to define directed cell transitions
- **ConnectivityKernel** (20%): uses scVI KNN graph edge weights for stability

**Why combine?** Pure velocity kernel is noisy; adding connectivity stabilises transitions for low-confidence velocity cells, reducing false bifurcations.

In [ ]:
print("Step 2 — Building combined velocity + connectivity kernel")

vk = VelocityKernel(adata)
vk.compute_transition_matrix()
print("  VelocityKernel computed")

ck = ConnectivityKernel(adata)
ck.compute_transition_matrix()
print("  ConnectivityKernel computed")

combined_kernel = 0.8 * vk + 0.2 * ck
combined_kernel.write_to_adata()
print("  Combined kernel (0.8 velocity + 0.2 connectivity) written to adata.obsp")

## Step 3 — GPCCA Macrostate Analysis

GPCCA (Generalized Perron Cluster Cluster Analysis) partitions the transition matrix into macrostates using Schur decomposition of the Markov chain.

- **Terminal states**: absorbing states (low outflow probability) — where cells end up
- **Initial states**: high-outflow source states — where trajectories begin
- **Transient states**: cells passing through on the way to terminal states

For RB, expected terminal states include: Subtype 1 (mature cone-like) and Subtype 2 (invasive/dedifferentiated).

In [ ]:
print(f"Step 3 — Running GPCCA (n_states={N_STATES})")

g = GPCCA(combined_kernel)
g.compute_schur(n_components=N_STATES + 1)
g.compute_macrostates(n_states=N_STATES, cluster_key="cell_type_broad")

if hasattr(g, "macrostates"):
    print(f"  Macrostates identified: {g.macrostates.cat.categories.tolist()}")

g.set_terminal_states_from_macrostates()
print(f"  Terminal states: {g.terminal_states.cat.categories.tolist()}")

## Step 4 — Compute Fate Probabilities

Fate probability P(cell → terminal state X) is computed via **absorption probability** in the Markov chain.

Each cell gets a probability vector over all terminal states that sums to 1. High probability toward the 'invasive' terminal state in extraocular cells would confirm the invasion model.

In [ ]:
print("Step 4 — Computing fate probabilities")

g.compute_fate_probabilities()

# Store fate probabilities in adata.obs
if hasattr(g, "fate_probabilities"):
    for state in g.terminal_states.cat.categories:
        col_name = f"fate_prob_{state.replace(' ', '_')}"
        adata.obs[col_name] = g.fate_probabilities[state].values
    fate_cols = [c for c in adata.obs.columns if c.startswith("fate_prob_")]
    print(f"  Fate probability columns: {fate_cols}")

# Store macrostate labels
adata.obs["macrostates_fwd"] = g.macrostates
if hasattr(g, "initial_states"):
    adata.obs["initial_states"] = g.initial_states

## Step 5 — Visualize Macrostates and Fate Probabilities

**Macrostate UMAP**: Each cell is coloured by its macrostate assignment. Check that initial states correspond to progenitor-like populations and terminal states to mature/invasive populations.

**Fate probability UMAP**: One panel per terminal state. Cells should show spatial continuity — high fate probability toward the invasive state should be enriched in known invasive subclusters.

In [ ]:
# Macrostate UMAP
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, key in zip(axes, ["macrostates_fwd", "initial_states"]):
    if key in adata.obs.columns:
        sc.pl.umap(adata, color=key, ax=ax, show=False, frameon=False,
                   size=3, alpha=0.8, title=key, legend_loc="right margin")
    else:
        ax.set_visible(False)
plt.suptitle("CellRank macrostates", fontsize=12, fontweight="bold")
plt.tight_layout()
fig.savefig(FIG_DIR / "cellrank_macrostates_umap.pdf", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Fate probability UMAPs
fate_cols = [c for c in adata.obs.columns if c.startswith("fate_prob_")]
if fate_cols:
    ncols = min(len(fate_cols), 4)
    nrows = int(np.ceil(len(fate_cols) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows))
    axes = np.array(axes).flatten() if nrows * ncols > 1 else [axes]

    for ax, key in zip(axes, fate_cols):
        sc.pl.umap(adata, color=key, ax=ax, show=False, frameon=False,
                   size=2, cmap="viridis", vmin=0, vmax=1,
                   title=key.replace("fate_prob_", "P(-> "))
    for ax in axes[len(fate_cols):]:
        ax.set_visible(False)

    plt.suptitle(
        "Fate probabilities (CellRank GPCCA)\nEach panel: P(cell reaches terminal state X)",
        fontsize=11, fontweight="bold", y=1.01,
    )
    plt.tight_layout()
    fig.savefig(FIG_DIR / "cellrank_fate_probabilities_umap.pdf", dpi=200, bbox_inches="tight")
    plt.show()

## Step 6 — Identify Lineage Driver Genes

Driver genes are those whose expression is most **correlated with the fate probability** of each terminal state.

- **Positive correlation**: induced in cells committing to that fate
- **Negative correlation**: repressed during commitment

These are candidate regulators of the invasion transition — potential therapeutic targets.

In [ ]:
print("Step 6 — Identifying lineage driver genes")

driver_dict = {}
for state in g.terminal_states.cat.categories:
    try:
        drivers = g.compute_lineage_drivers(
            lineages=state,
            cluster_key="cell_type_broad",
            use_raw=False,
            return_drivers=True,
        )
        if drivers is not None:
            driver_dict[state] = drivers
            print(f"  Top 5 drivers -> {state}: {drivers.head(5).index.tolist()}")
    except Exception as e:
        print(f"  Driver computation for {state} failed: {e}")

# Save driver gene tables
if driver_dict:
    all_drivers = pd.concat(
        [df.assign(terminal_state=s) for s, df in driver_dict.items()]
    )
    all_drivers.to_csv(TAB_DIR / "cellrank_driver_genes.csv")
    print(f"  Saved driver genes -> {TAB_DIR / 'cellrank_driver_genes.csv'}")

## Step 7 — Fate Probabilities by Disease Stage

**Key test**: Are extraocular tumour cells enriched for the invasive terminal fate probability?  
Expected: violin plots should show significantly higher fate probability toward the invasive state in extraocular vs. intraocular samples.

In [ ]:
fate_cols = [c for c in adata.obs.columns if c.startswith("fate_prob_")]

if "disease_stage" in adata.obs.columns and fate_cols:
    fig, axes = plt.subplots(1, len(fate_cols), figsize=(6 * len(fate_cols), 5))
    if len(fate_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, fate_cols):
        sc.pl.violin(
            adata, keys=col, groupby="disease_stage",
            ax=ax, show=False, rotation=30,
            title=col.replace("fate_prob_", "P(-> "),
        )
    plt.suptitle("Fate probabilities: intraocular vs. extraocular",
                 fontsize=11, fontweight="bold")
    plt.tight_layout()
    fig.savefig(FIG_DIR / "cellrank_fate_probs_by_disease_stage.pdf",
                dpi=200, bbox_inches="tight")
    plt.show()
else:
    print("  'disease_stage' column not found — skipping comparison plot")
    print("  Add disease_stage annotation to adata.obs for this comparison")

## Step 8 — Save Fate Tables and Atlas

In [ ]:
# Save fate probability table
fate_cols = [c for c in adata.obs.columns if c.startswith("fate_prob_")]
meta_cols = ["sample_id", "dataset", "cell_type_broad", "rb_subtype", "velocity_pseudotime"]
meta_cols = [c for c in meta_cols if c in adata.obs.columns]
if fate_cols:
    adata.obs[meta_cols + fate_cols].to_csv(TAB_DIR / "cellrank_fate_probabilities.csv")
    print(f"Saved fate probabilities -> {TAB_DIR / 'cellrank_fate_probabilities.csv'}")

# Save atlas
adata.uns["cellrank_terminal_states"] = g.terminal_states.cat.categories.tolist()
print(f"\nSaving CellRank-annotated atlas -> {OUT_H5AD.name}")
adata.write_h5ad(OUT_H5AD, compression="gzip")
size_mb = OUT_H5AD.stat().st_size / 1e6
print(f"  {adata.n_obs:,} cells x {adata.n_vars:,} genes | {size_mb:.0f} MB")
print("  Added:")
print("    .obs['macrostates_fwd']   : GPCCA macrostate labels")
print("    .obs['fate_prob_*']       : per-terminal-state fate probabilities")
print("    .obsp['T_fwd']            : forward transition matrix")
print("  Next: Run notebook 09_ligand_receptor_communication.ipynb")